# 02 — Model Development

**Wearing the Model Development hat.** Trains a logistic regression scorecard on the WoE-transformed features. This notebook intentionally does NOT contain validation logic — that is performed independently in `03_independent_validation.ipynb`, mirroring the real separation between first-line (development) and second-line (validation) functions.

In [ ]:
import sys
sys.path.append('../src')
import pandas as pd
from model_training import train_pd_model, score_dataset, coefficient_report, WOE_FEATURES
import joblib

In [ ]:
train = pd.read_csv('../data/processed/train_woe.csv')
test = pd.read_csv('../data/processed/test_woe.csv')
oot = pd.read_csv('../data/processed/oot_woe.csv')

## Fit the scorecard

In [ ]:
model = train_pd_model(train)
print('Model fitted.')

## Development team self-check: coefficient signs

Since `SeriousDlqin2yrs=1` means default, and WoE = ln(%good/%bad), every coefficient on a WoE feature is expected to be **negative** (a safer bin should reduce the log-odds of default).

In [ ]:
coef_df = coefficient_report(model, WOE_FEATURES)
coef_df

**Observation:** Two variables (`NumberOfTimes90DaysLate_woe`, `NumberOfTime60-89DaysPastDueNotWorse_woe`) show a coefficient of exactly zero. This is investigated and documented as a formal finding in the validation notebook.

## Score all samples and persist

In [ ]:
train['predicted_pd'] = score_dataset(model, train)
test['predicted_pd'] = score_dataset(model, test)
oot['predicted_pd'] = score_dataset(model, oot)

train.to_csv('../data/processed/train_scored.csv', index=False)
test.to_csv('../data/processed/test_scored.csv', index=False)
oot.to_csv('../data/processed/oot_scored.csv', index=False)
joblib.dump(model, '../data/processed/pd_model.pkl')
print('Model and scored datasets saved.')